In [1]:
import pandas as pd
from sklearn.svm import LinearSVC
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
#Load Dataset
df = pd.read_csv("dataset.csv")
df.head(50)

,Stars,Title_of_Review,Base_Review,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,1.0 out of 5 stars,This is a ripoff. And shame on Amazon for not ...,Don't buy this product. I purchased it for my ...,NaN,NaN,NaN
1,1.0 out of 5 stars,PDF Max Pro doesn't play well with marshmallow,"Get an ""invalid file"" error since Marshmallow.",NaN,NaN,NaN
2,1.0 out of 5 stars,Worse app ever,I wouldn't even give it a one if I could. I wi...,NaN,NaN,NaN
3,1.0 out of 5 stars,I'd recommend trying to email the developer fi...,"If I could give it less stars, I would! Does n...",NaN,NaN,NaN
4,5.0 out of 5 stars,Amazing For Markup,I have several apps that can open pdf files on...,NaN,NaN,NaN
5,2.0 out of 5 stars,Tools work great but loses files,The app works great for reading and annotating...,NaN,NaN,NaN
6,1.0 out of 5 stars,Doesn't Work At All,"Wow... PDF Files are ""invalid"" and cannot be ...",NaN,NaN,NaN
7,3.0 out of 5 stars,"Alright, but EZ PDF is better. (One Reason Only)",This app is good at its job. There is still on...,NaN,NaN,NaN
8,3.0 out of 5 stars,College Student review,Has a lot of good features. I'm not going to g...,NaN,NaN,NaN
9,3.0 out of 5 stars,Nice app except for one BIG thing,This would be a fantastic app except for one b...,NaN,NaN,NaN


In [3]:
df.drop(columns=["Title_of_Review","Unnamed: 3", "Unnamed: 4", "Unnamed: 5"], inplace=True)
df.head()

,Stars,Base_Review
0,1.0 out of 5 stars,Don't buy this product. I purchased it for my ...
1,1.0 out of 5 stars,"Get an ""invalid file"" error since Marshmallow."
2,1.0 out of 5 stars,I wouldn't even give it a one if I could. I wi...
3,1.0 out of 5 stars,"If I could give it less stars, I would! Does n..."
4,5.0 out of 5 stars,I have several apps that can open pdf files on...


In [4]:
df.dropna(inplace=True)
df.isnull().sum()

Stars          0
Base_Review    0
dtype: int64

In [5]:
df['Stars'].value_counts()

Stars
5.0 out of 5 stars    33246
1.0 out of 5 stars    21692
4.0 out of 5 stars    11603
3.0 out of 5 stars     7666
2.0 out of 5 stars     5280
a-icon-alt                1
Name: count, dtype: int64

In [6]:
df['Base_Review'] = df['Base_Review'].str.lower().str.strip()

In [7]:
noise_text = "Translate review to English"
df = df[df['Base_Review'] != noise_text]

In [8]:
df['Base_Review'] = df['Base_Review'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

In [9]:
df = df.drop_duplicates(subset=['Base_Review', 'Stars'])

In [10]:
df = df[df['Stars'] != 'a-icon-alt'] #removes unnecessary text in the Stars column

In [11]:
df['rating'] = df['Stars'].apply(lambda x: int(float(str(x).split()[0])))
df = df.drop('Stars', axis=1)

In [12]:
X = df['Base_Review']
y = df['rating']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [14]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2)) #ignore common stop words that do not help the review. Keep the most important 5000 words
#ngrams to detect pairs of words that often appear together
X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

In [15]:
svm_model = LinearSVC( 
    class_weight='balanced', 
    random_state=42, 
    max_iter=2000
)

svm_model.fit(X_train_tfidf, y_train)

y_pred_svm = svm_model.predict(X_test_tfidf)

In [16]:
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.2f}")
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.56
              precision    recall  f1-score   support

           1       0.65      0.73      0.69      3554
           2       0.14      0.17      0.15       913
           3       0.23      0.24      0.24      1249
           4       0.37      0.30      0.33      2116
           5       0.73      0.70      0.72      5411

    accuracy                           0.56     13243
   macro avg       0.43      0.43      0.43     13243
weighted avg       0.57      0.56      0.56     13243



In [18]:
import pickle

with open("predictor.pkl", "rb") as f:
    model = pickle.load(f)

with open("tfidf.pkl", "rb") as f:
    vectorizer = pickle.load(f)